# Hypothesentests & Prädiktives Modell

Dieses Notebook testet vier Hypothesen zum Bank-Telemarketing-Datensatz und erstellt ein Klassifikationsmodell zur Vorhersage der Zielvariable `y` (Abschluss einer Termineinlage).

**Hypothesen:**
1. Je mehr Kontaktversuche in dieser Kampagne (`campaign`), desto unwahrscheinlicher der Abschluss.
2. Kunden mit Immobilienkredit (`housing=yes`) UND persönlichem Kredit (`loan=yes`) schließen seltener ab.
3. Kunden, bei denen die vorherige Kampagne erfolgreich war (`poutcome=success`), schließen signifikant häufiger ab.
4. Kunden in Berufsgruppen mit höherem/gesichertem Einkommen (z.B. Management, Rentner) schließen signifikant häufiger ab als Blue-Collar-Arbeiter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, pointbiserialr

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, accuracy_score, f1_score, precision_recall_curve,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Setup erfolgreich.')

In [ ]:
# Rohdaten laden für Hypothesentests (auf Originalskala)
raw_data = pd.read_csv('bank-full.csv', sep=';')

# Binäre Zielvariable erstellen
raw_data['y_binary'] = (raw_data['y'] == 'yes').astype(int)

print(f'Datensatz geladen: {raw_data.shape[0]} Zeilen, {raw_data.shape[1]} Spalten')
print(f'\nZielvariable y:')
print(raw_data['y'].value_counts())
print(f'\nAbschlussrate: {raw_data["y_binary"].mean():.2%}')

---
## Hypothese 1: Kampagnen-Kontaktversuche und Abschlusswahrscheinlichkeit

**H0:** Die Anzahl der Kontaktversuche (`campaign`) hat keinen Einfluss auf die Abschlusswahrscheinlichkeit.

**H1:** Je mehr Kontaktversuche in dieser Kampagne, desto unwahrscheinlicher der Abschluss.

**Methode:** Mann-Whitney-U-Test (da `campaign` nicht normalverteilt ist) + Punkt-biseriale Korrelation

In [ ]:
# Gruppen aufteilen
campaign_yes = raw_data[raw_data['y'] == 'yes']['campaign']
campaign_no = raw_data[raw_data['y'] == 'no']['campaign']

print('=== Hypothese 1: Campaign vs. Abschluss ===')
print(f'\nDeskriptive Statistik:')
print(f'  Abschluss (yes): Median={campaign_yes.median():.0f}, Mean={campaign_yes.mean():.2f}, Std={campaign_yes.std():.2f}')
print(f'  Kein Abschluss (no): Median={campaign_no.median():.0f}, Mean={campaign_no.mean():.2f}, Std={campaign_no.std():.2f}')

# Mann-Whitney-U-Test (einseitig: campaign_no > campaign_yes erwartet)
u_stat, p_value_two = mannwhitneyu(campaign_no, campaign_yes, alternative='two-sided')
_, p_value_greater = mannwhitneyu(campaign_no, campaign_yes, alternative='greater')

print(f'\nMann-Whitney-U-Test:')
print(f'  U-Statistik: {u_stat:,.0f}')
print(f'  p-Wert (zweiseitig): {p_value_two:.2e}')
print(f'  p-Wert (einseitig, no > yes): {p_value_greater:.2e}')

# Punkt-biseriale Korrelation
r_pb, p_pb = pointbiserialr(raw_data['y_binary'], raw_data['campaign'])
print(f'\nPunkt-biseriale Korrelation:')
print(f'  r = {r_pb:.4f}, p = {p_pb:.2e}')

# Effektstärke (rank-biserial correlation aus U-Statistik)
n1, n2 = len(campaign_no), len(campaign_yes)
r_rb = 1 - (2 * u_stat) / (n1 * n2)
print(f'\nEffektstärke (rank-biserial r): {r_rb:.4f}')

alpha = 0.05
if p_value_greater < alpha:
    print(f'\n=> H0 wird abgelehnt (p={p_value_greater:.2e} < {alpha}).')
    print('   ERGEBNIS: Kunden ohne Abschluss haben signifikant mehr Kontaktversuche.')
    print('   Die Hypothese wird best\u00e4tigt.')
else:
    print(f'\n=> H0 kann nicht abgelehnt werden (p={p_value_greater:.2e} >= {alpha}).')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Boxplot
sns.boxplot(x='y', y='campaign', data=raw_data, ax=axes[0], palette='Set2')
axes[0].set_title('Campaign nach Abschluss (y)')
axes[0].set_xlabel('Abschluss')
axes[0].set_ylabel('Kontaktversuche (campaign)')
axes[0].set_ylim(0, 15)  # Begrenzung für bessere Lesbarkeit

# Abschlussrate nach campaign-Gruppen
raw_data['campaign_group'] = pd.cut(raw_data['campaign'], bins=[0, 1, 2, 3, 5, 10, 100],
                                     labels=['1', '2', '3', '4-5', '6-10', '>10'])
rate_by_campaign = raw_data.groupby('campaign_group', observed=False)['y_binary'].mean()

rate_by_campaign.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2'), edgecolor='black')
axes[1].set_title('Abschlussrate nach Kontaktversuche-Gruppe')
axes[1].set_xlabel('Kontaktversuche')
axes[1].set_ylabel('Abschlussrate')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(rate_by_campaign):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=9)

# Violin-Plot
sns.violinplot(x='y', y='campaign', data=raw_data[raw_data['campaign'] <= 15],
               ax=axes[2], palette='Set2', inner='quartile')
axes[2].set_title('Verteilung Campaign (bis 15)')
axes[2].set_xlabel('Abschluss')
axes[2].set_ylabel('Kontaktversuche')

plt.tight_layout()
plt.show()

raw_data = raw_data.drop('campaign_group', axis=1)

---
## Hypothese 2: Doppelbelastung durch Housing + Loan

**H0:** Der Abschluss ist unabhängig davon, ob ein Kunde sowohl Immobilienkredit als auch Privatkredit hat.

**H1:** Kunden mit Immobilienkredit (`housing=yes`) UND persönlichem Kredit (`loan=yes`) schließen seltener ab.

**Methode:** Chi²-Unabhängigkeitstest + Vergleich der Abschlussraten

In [ ]:
# Interaktionsvariable erstellen
raw_data['housing_loan'] = 'andere'
raw_data.loc[(raw_data['housing'] == 'yes') & (raw_data['loan'] == 'yes'), 'housing_loan'] = 'beide_ja'
raw_data.loc[(raw_data['housing'] == 'yes') & (raw_data['loan'] == 'no'), 'housing_loan'] = 'nur_housing'
raw_data.loc[(raw_data['housing'] == 'no') & (raw_data['loan'] == 'yes'), 'housing_loan'] = 'nur_loan'
raw_data.loc[(raw_data['housing'] == 'no') & (raw_data['loan'] == 'no'), 'housing_loan'] = 'keine_kredite'

print('=== Hypothese 2: Housing + Loan vs. Abschluss ===')
print(f'\nAbschlussraten nach Kreditkombination:')
rates = raw_data.groupby('housing_loan')['y_binary'].agg(['mean', 'count', 'sum'])
rates.columns = ['Abschlussrate', 'Anzahl', 'Abschl\u00fcsse']
rates = rates.sort_values('Abschlussrate')
print(rates.to_string())

# Chi²-Test: housing_loan vs. y
contingency = pd.crosstab(raw_data['housing_loan'], raw_data['y'])
chi2, p_chi2, dof, expected = chi2_contingency(contingency)

print(f'\nChi\u00b2-Unabh\u00e4ngigkeitstest:')
print(f'  Chi\u00b2-Statistik: {chi2:.2f}')
print(f'  Freiheitsgrade: {dof}')
print(f'  p-Wert: {p_chi2:.2e}')

# Cramers V als Effektstärke
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
print(f'  Cram\u00e9rs V: {cramers_v:.4f}')

# Gezielter Vergleich: beide_ja vs. keine_kredite
both = raw_data[raw_data['housing_loan'] == 'beide_ja']['y_binary']
neither = raw_data[raw_data['housing_loan'] == 'keine_kredite']['y_binary']

# Proportionstest (z-Test für zwei Proportionen)
p1, p2 = both.mean(), neither.mean()
n1, n2 = len(both), len(neither)
p_pool = (both.sum() + neither.sum()) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z_stat = (p1 - p2) / se
p_z = stats.norm.sf(abs(z_stat)) * 2  # zweiseitig

print(f'\nProportionstest (beide_ja vs. keine_kredite):')
print(f'  Rate beide Kredite: {p1:.4f}')
print(f'  Rate keine Kredite: {p2:.4f}')
print(f'  Differenz: {p1 - p2:.4f}')
print(f'  z-Statistik: {z_stat:.4f}')
print(f'  p-Wert: {p_z:.2e}')

if p_chi2 < 0.05:
    print(f'\n=> H0 wird abgelehnt (p={p_chi2:.2e} < 0.05).')
    print('   ERGEBNIS: Es besteht ein signifikanter Zusammenhang zwischen der')
    print('   Kreditkombination und dem Abschluss.')
    if p1 < p2:
        print('   Die Hypothese wird best\u00e4tigt: Kunden mit beiden Krediten schlie\u00dfen seltener ab.')
else:
    print(f'\n=> H0 kann nicht abgelehnt werden (p={p_chi2:.2e} >= 0.05).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Abschlussrate nach Kreditkombination
order = ['keine_kredite', 'nur_loan', 'nur_housing', 'beide_ja']
rate_data = raw_data.groupby('housing_loan')['y_binary'].mean().reindex(order)

colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']
bars = rate_data.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Abschlussrate nach Kreditkombination')
axes[0].set_xlabel('Kreditkombination')
axes[0].set_ylabel('Abschlussrate')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(rate_data):
    axes[0].text(i, v + 0.003, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')

# Gestapeltes Balkendiagramm (Anteile)
ct_norm = pd.crosstab(raw_data['housing_loan'], raw_data['y'], normalize='index').reindex(order)
ct_norm.plot(kind='bar', stacked=True, ax=axes[1], color=['#ef8a62', '#67a9cf'], edgecolor='black')
axes[1].set_title('Anteil Abschluss/Kein Abschluss nach Kreditkombination')
axes[1].set_xlabel('Kreditkombination')
axes[1].set_ylabel('Anteil')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(title='Abschluss')

plt.tight_layout()
plt.show()

---
## Hypothese 3: Erfolg der vorherigen Kampagne

**H0:** Der Erfolg der vorherigen Kampagne (`poutcome`) hat keinen Einfluss auf den Abschluss.

**H1:** Kunden, bei denen die vorherige Kampagne erfolgreich war (`poutcome=success`), schließen signifikant häufiger ab.

**Methode:** Chi²-Test + Proportionstest (success vs. nicht-success)

In [ ]:
print('=== Hypothese 3: Poutcome vs. Abschluss ===')

# Abschlussraten nach poutcome
print(f'\nAbschlussraten nach poutcome:')
poutcome_rates = raw_data.groupby('poutcome')['y_binary'].agg(['mean', 'count', 'sum'])
poutcome_rates.columns = ['Abschlussrate', 'Anzahl', 'Abschl\u00fcsse']
poutcome_rates = poutcome_rates.sort_values('Abschlussrate', ascending=False)
print(poutcome_rates.to_string())

# Chi²-Test für gesamte poutcome vs. y
contingency_pout = pd.crosstab(raw_data['poutcome'], raw_data['y'])
chi2_p, p_chi2_p, dof_p, _ = chi2_contingency(contingency_pout)

n_total = contingency_pout.sum().sum()
cramers_v_p = np.sqrt(chi2_p / (n_total * (min(contingency_pout.shape) - 1)))

print(f'\nChi\u00b2-Test (poutcome vs. y):')
print(f'  Chi\u00b2-Statistik: {chi2_p:.2f}')
print(f'  Freiheitsgrade: {dof_p}')
print(f'  p-Wert: {p_chi2_p:.2e}')
print(f'  Cram\u00e9rs V: {cramers_v_p:.4f}')

# Gezielter Vergleich: success vs. alle anderen (ohne unknown)
known_poutcome = raw_data[raw_data['poutcome'] != 'unknown']
success_group = known_poutcome[known_poutcome['poutcome'] == 'success']['y_binary']
other_group = known_poutcome[known_poutcome['poutcome'] != 'success']['y_binary']

p_s, p_o = success_group.mean(), other_group.mean()
n_s, n_o = len(success_group), len(other_group)
p_pool2 = (success_group.sum() + other_group.sum()) / (n_s + n_o)
se2 = np.sqrt(p_pool2 * (1 - p_pool2) * (1/n_s + 1/n_o))
z_stat2 = (p_s - p_o) / se2
p_z2 = stats.norm.sf(abs(z_stat2))  # einseitig

print(f'\nProportionstest (success vs. andere, ohne unknown):')
print(f'  Rate poutcome=success: {p_s:.4f} (n={n_s})')
print(f'  Rate poutcome=failure/other: {p_o:.4f} (n={n_o})')
print(f'  z-Statistik: {z_stat2:.4f}')
print(f'  p-Wert (einseitig): {p_z2:.2e}')

# Odds Ratio
odds_success = p_s / (1 - p_s)
odds_other = p_o / (1 - p_o)
odds_ratio = odds_success / odds_other
print(f'\n  Odds Ratio (success vs. andere): {odds_ratio:.2f}')
print(f'  => Kunden mit poutcome=success haben eine {odds_ratio:.1f}x h\u00f6here Chance abzuschlie\u00dfen.')

if p_z2 < 0.05:
    print(f'\n=> H0 wird abgelehnt (p={p_z2:.2e} < 0.05).')
    print('   ERGEBNIS: Kunden mit erfolgreichem Vorkampagnen-Ergebnis schlie\u00dfen')
    print('   signifikant h\u00e4ufiger ab. Die Hypothese wird best\u00e4tigt.')
else:
    print(f'\n=> H0 kann nicht abgelehnt werden (p={p_z2:.2e} >= 0.05).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Abschlussrate nach poutcome
order_pout = ['success', 'other', 'failure', 'unknown']
rate_pout = raw_data.groupby('poutcome')['y_binary'].mean().reindex(order_pout)

colors_pout = ['#2ca02c', '#ff7f0e', '#d62728', '#7f7f7f']
rate_pout.plot(kind='bar', ax=axes[0], color=colors_pout, edgecolor='black')
axes[0].set_title('Abschlussrate nach Ergebnis der vorherigen Kampagne')
axes[0].set_xlabel('poutcome')
axes[0].set_ylabel('Abschlussrate')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(rate_pout):
    axes[0].text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')

# Mosaikplot-ähnliches gestapeltes Diagramm
ct_pout = pd.crosstab(raw_data['poutcome'], raw_data['y'], normalize='index').reindex(order_pout)
ct_pout.plot(kind='bar', stacked=True, ax=axes[1], color=['#ef8a62', '#67a9cf'], edgecolor='black')
axes[1].set_title('Anteil Abschluss/Kein Abschluss nach poutcome')
axes[1].set_xlabel('poutcome')
axes[1].set_ylabel('Anteil')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Abschluss')

plt.tight_layout()
plt.show()

---
## Hypothese 4: Berufsgruppen und Abschlusswahrscheinlichkeit

**H0:** Die Berufsgruppe hat keinen Einfluss auf die Abschlusswahrscheinlichkeit.

**H1:** Kunden in Berufsgruppen mit höherem/gesichertem Einkommen (Management, Rentner) schließen signifikant häufiger ab als Blue-Collar-Arbeiter.

**Methode:** Chi²-Test + paarweise Proportions-Tests (Management/Retired vs. Blue-Collar)

In [ ]:
print('=== Hypothese 4: Berufsgruppen vs. Abschluss ===')

# Abschlussraten nach Beruf
print(f'\nAbschlussraten nach Beruf:')
job_rates = raw_data.groupby('job')['y_binary'].agg(['mean', 'count', 'sum'])
job_rates.columns = ['Abschlussrate', 'Anzahl', 'Abschl\u00fcsse']
job_rates = job_rates.sort_values('Abschlussrate', ascending=False)
print(job_rates.to_string())

# Chi²-Test über alle Berufsgruppen
contingency_job = pd.crosstab(raw_data['job'], raw_data['y'])
chi2_j, p_chi2_j, dof_j, _ = chi2_contingency(contingency_job)

n_j = contingency_job.sum().sum()
cramers_v_j = np.sqrt(chi2_j / (n_j * (min(contingency_job.shape) - 1)))

print(f'\nChi\u00b2-Test (job vs. y):')
print(f'  Chi\u00b2-Statistik: {chi2_j:.2f}')
print(f'  Freiheitsgrade: {dof_j}')
print(f'  p-Wert: {p_chi2_j:.2e}')
print(f'  Cram\u00e9rs V: {cramers_v_j:.4f}')

# Paarweise Tests gegen blue-collar
blue_collar = raw_data[raw_data['job'] == 'blue-collar']['y_binary']
compare_jobs = ['management', 'retired', 'student']

print(f'\nPaarweise Vergleiche gegen blue-collar (Rate: {blue_collar.mean():.4f}):')
for job_name in compare_jobs:
    group = raw_data[raw_data['job'] == job_name]['y_binary']
    p1, p2 = group.mean(), blue_collar.mean()
    n1, n2 = len(group), len(blue_collar)
    p_pool_j = (group.sum() + blue_collar.sum()) / (n1 + n2)
    se_j = np.sqrt(p_pool_j * (1 - p_pool_j) * (1/n1 + 1/n2))
    z_j = (p1 - p2) / se_j
    p_j = stats.norm.sf(z_j)  # einseitig: höhere Rate erwartet
    
    # Odds Ratio
    odds1 = p1 / (1 - p1)
    odds2 = p2 / (1 - p2)
    or_j = odds1 / odds2
    
    sig = '***' if p_j < 0.001 else '**' if p_j < 0.01 else '*' if p_j < 0.05 else 'n.s.'
    print(f'  {job_name:15s}: Rate={p1:.4f}, z={z_j:.2f}, p={p_j:.2e}, OR={or_j:.2f} {sig}')

if p_chi2_j < 0.05:
    print(f'\n=> H0 wird abgelehnt (p={p_chi2_j:.2e} < 0.05).')
    print('   ERGEBNIS: Es besteht ein signifikanter Zusammenhang zwischen Beruf und Abschluss.')
    print('   Management, Rentner und Studenten schlie\u00dfen h\u00e4ufiger ab als Blue-Collar.')
    print('   Die Hypothese wird best\u00e4tigt.')
else:
    print(f'\n=> H0 kann nicht abgelehnt werden (p={p_chi2_j:.2e} >= 0.05).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Abschlussrate pro Berufsgruppe
job_rate_sorted = raw_data.groupby('job')['y_binary'].mean().sort_values(ascending=True)

# Farben: Highlight für blue-collar, management, retired
colors_job = []
for job in job_rate_sorted.index:
    if job == 'blue-collar':
        colors_job.append('#d62728')  # rot
    elif job in ['management', 'retired']:
        colors_job.append('#2ca02c')  # grün
    else:
        colors_job.append('#7f7f7f')  # grau

job_rate_sorted.plot(kind='barh', ax=axes[0], color=colors_job, edgecolor='black')
axes[0].set_title('Abschlussrate nach Berufsgruppe')
axes[0].set_xlabel('Abschlussrate')
axes[0].set_ylabel('Beruf')
for i, v in enumerate(job_rate_sorted):
    axes[0].text(v + 0.003, i, f'{v:.1%}', va='center', fontsize=9)

# Gruppierter Vergleich: High-Income vs. Blue-Collar
raw_data['job_category'] = raw_data['job'].apply(
    lambda x: 'High-Income/Secured' if x in ['management', 'retired', 'student'] 
    else ('Blue-Collar' if x == 'blue-collar' else 'Andere')
)
cat_rates = raw_data.groupby('job_category')['y_binary'].mean().reindex(
    ['High-Income/Secured', 'Andere', 'Blue-Collar'])

cat_rates.plot(kind='bar', ax=axes[1], color=['#2ca02c', '#7f7f7f', '#d62728'], edgecolor='black')
axes[1].set_title('Abschlussrate: High-Income vs. Blue-Collar')
axes[1].set_xlabel('Berufskategorie')
axes[1].set_ylabel('Abschlussrate')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(cat_rates):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

raw_data = raw_data.drop('job_category', axis=1)

---
# Prädiktives Modell

Aufbauend auf den Hypothesentests erstellen wir ein Klassifikationsmodell. Wir verwenden:
1. **Logistische Regression** - gut interpretierbar, liefert Odds Ratios für die Hypothesenvalidierung
2. **Random Forest** - robustes Ensemble-Modell als Vergleich
3. **Gradient Boosting** - leistungsstarkes Modell als Benchmark

**Wichtig:** Die Variable `duration` wird entfernt, da sie erst nach dem Gespräch bekannt ist und somit keinen prädiktiven Wert vor dem Anruf hat.

In [ ]:
# ========================================
# Datenaufbereitung für das Modell
# ========================================

model_data = raw_data.copy()

# Duration entfernen (erst nach dem Gespräch bekannt)
model_data = model_data.drop('duration', axis=1)

# Temporäre Spalten entfernen
if 'housing_loan' in model_data.columns:
    model_data = model_data.drop('housing_loan', axis=1)
if 'y_binary' in model_data.columns:
    pass  # behalten als Zielvariable
if 'y' in model_data.columns:
    model_data = model_data.drop('y', axis=1)

# ========================================
# Feature Engineering: Hypothesen-spezifische Features
# ========================================

# H2: Interaktion Housing x Loan
model_data['housing_AND_loan'] = ((model_data['housing'] == 'yes') & 
                                   (model_data['loan'] == 'yes')).astype(int)

# H4: Job-Kategorie (High-Income vs. Blue-Collar)
model_data['job_high_income'] = model_data['job'].isin(['management', 'retired']).astype(int)
model_data['job_blue_collar'] = (model_data['job'] == 'blue-collar').astype(int)

# ========================================
# Encoding
# ========================================

# Kategoriale Spalten
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
model_data = pd.get_dummies(model_data, columns=cat_cols, drop_first=True)

# Numerische Features: log-Transformation für schiefe Variablen
model_data['campaign_log'] = np.log1p(model_data['campaign'])
model_data['balance_log'] = np.log1p(model_data['balance'] - model_data['balance'].min() + 1)

# Originale Spalten entfernen nach Transformation
model_data = model_data.drop(['campaign', 'balance'], axis=1)

# Boolean-Spalten zu int konvertieren
bool_cols = model_data.select_dtypes(include=['bool']).columns
model_data[bool_cols] = model_data[bool_cols].astype(int)

print(f'Feature-Matrix: {model_data.shape[0]} Zeilen, {model_data.shape[1] - 1} Features')
print(f'Zielvariable (y_binary): {model_data["y_binary"].value_counts().to_dict()}')
print(f'\nFeatures:')
feature_cols = [c for c in model_data.columns if c != 'y_binary']
print(feature_cols)

In [ ]:
# ========================================
# Train/Test Split
# ========================================

X = model_data.drop('y_binary', axis=1)
y = model_data['y_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training Set: {X_train.shape[0]} Zeilen ({y_train.mean():.2%} positiv)')
print(f'Test Set:     {X_test.shape[0]} Zeilen ({y_test.mean():.2%} positiv)')

# Skalierung
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE für das Training (Class Imbalance)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'\nNach SMOTE: {len(X_train_resampled)} Zeilen')
print(f'  Klasse 0: {(y_train_resampled == 0).sum()}')
print(f'  Klasse 1: {(y_train_resampled == 1).sum()}')

In [ ]:
# ========================================
# Modelle trainieren
# ========================================

models = {
    'Logistische Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, 
                                            random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                     learning_rate=0.1, random_state=42)
}

results = {}
for name, model in models.items():
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')
    
    # Training auf SMOTE-resampled Daten
    model.fit(X_train_resampled, y_train_resampled)
    
    # Vorhersagen auf dem Test-Set
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metriken
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'accuracy': acc,
        'f1_score': f1,
        'roc_auc': roc_auc
    }
    
    print(f'\nAccuracy:  {acc:.4f}')
    print(f'F1-Score:  {f1:.4f}')
    print(f'ROC-AUC:   {roc_auc:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Kein Abschluss', 'Abschluss']))

In [ ]:
# ========================================
# Modellvergleich
# ========================================

print('\n' + '='*60)
print('MODELLVERGLEICH')
print('='*60)

comparison_df = pd.DataFrame({
    'Modell': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'F1-Score': [r['f1_score'] for r in results.values()],
    'ROC-AUC': [r['roc_auc'] for r in results.values()]
}).set_index('Modell')

print(comparison_df.to_string())

# Metriken-Vergleich visualisieren
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['Accuracy', 'F1-Score', 'ROC-AUC']
colors_m = ['#66c2a5', '#fc8d62', '#8da0cb']

for i, metric in enumerate(metrics):
    bars = comparison_df[metric].plot(kind='bar', ax=axes[i], color=colors_m, edgecolor='black')
    axes[i].set_title(metric)
    axes[i].set_ylabel(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=25)
    for j, v in enumerate(comparison_df[metric]):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# Confusion Matrices und ROC-Kurven
# ========================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, (name, res) in enumerate(results.items()):
    # Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(
        y_test, res['y_pred'], 
        display_labels=['Kein Abschluss', 'Abschluss'],
        ax=axes[0, i], cmap='Blues'
    )
    axes[0, i].set_title(f'{name}\nConfusion Matrix')
    
    # ROC-Kurve
    RocCurveDisplay.from_predictions(
        y_test, res['y_pred_proba'],
        ax=axes[1, i], color=colors_m[i]
    )
    axes[1, i].set_title(f'{name}\nROC-Kurve (AUC={res["roc_auc"]:.3f})')
    axes[1, i].plot([0, 1], [0, 1], 'k--', alpha=0.5)

plt.tight_layout()
plt.show()

## Feature Importance & Koeffizienten-Analyse

Die logistische Regression liefert direkt interpretierbare Koeffizienten (als Odds Ratios), während der Random Forest Feature-Importance-Werte liefert.

In [ ]:
# ========================================
# Logistische Regression: Koeffizienten (Odds Ratios)
# ========================================

lr_model = results['Logistische Regression']['model']
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Koeffizient': lr_model.coef_[0],
    'Odds Ratio': np.exp(lr_model.coef_[0])
}).sort_values('Koeffizient', ascending=False)

print('=== Top 15 st\u00e4rkste positive Einfl\u00fcsse (Logistische Regression) ===')
print(coef_df.head(15).to_string(index=False))

print('\n=== Top 15 st\u00e4rkste negative Einfl\u00fcsse ===')
print(coef_df.tail(15).to_string(index=False))

# ========================================
# Hypothesenrelevante Koeffizienten hervorheben
# ========================================

print('\n' + '='*60)
print('HYPOTHESENRELEVANTE KOEFFIZIENTEN')
print('='*60)

hypothesis_features = {
    'H1 (Campaign)': 'campaign_log',
    'H2 (Housing AND Loan)': 'housing_AND_loan',
    'H3 (Poutcome Success)': 'poutcome_success',
    'H4a (Job High Income)': 'job_high_income',
    'H4b (Job Blue-Collar)': 'job_blue_collar'
}

for hyp_name, feat_name in hypothesis_features.items():
    if feat_name in coef_df['Feature'].values:
        row = coef_df[coef_df['Feature'] == feat_name].iloc[0]
        direction = 'positiv' if row['Koeffizient'] > 0 else 'negativ'
        print(f'  {hyp_name:30s}: Koeff={row["Koeffizient"]:+.4f}, OR={row["Odds Ratio"]:.4f} ({direction})')
    else:
        print(f'  {hyp_name:30s}: Feature nicht im Modell gefunden')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# ========================================
# Logistische Regression: Top 20 Koeffizienten
# ========================================
top_n = 20
top_positive = coef_df.head(top_n // 2)
top_negative = coef_df.tail(top_n // 2)
top_features = pd.concat([top_positive, top_negative]).sort_values('Koeffizient')

colors_coef = ['#d62728' if v < 0 else '#2ca02c' for v in top_features['Koeffizient']]
top_features.set_index('Feature')['Koeffizient'].plot(
    kind='barh', ax=axes[0], color=colors_coef, edgecolor='black'
)
axes[0].set_title('Logistische Regression: Top Koeffizienten')
axes[0].set_xlabel('Koeffizient (log-odds)')
axes[0].axvline(x=0, color='black', linewidth=0.8)

# ========================================
# Random Forest: Feature Importance
# ========================================
rf_model = results['Random Forest']['model']
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True).tail(top_n)

rf_importance.set_index('Feature')['Importance'].plot(
    kind='barh', ax=axes[1], color='#1f77b4', edgecolor='black'
)
axes[1].set_title('Random Forest: Top Feature Importance')
axes[1].set_xlabel('Feature Importance')

plt.tight_layout()
plt.show()

## Kreuzvalidierung

Um die Robustheit der Modelle zu bewerten, führen wir eine 5-fache stratifizierte Kreuzvalidierung durch.

In [ ]:
# ========================================
# 5-Fold Stratified Cross-Validation
# ========================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Auf skalierten, nicht-SMOTE Daten (SMOTE würde Data Leakage verursachen)
X_all_scaled = scaler.fit_transform(X)

print('5-Fold Stratified Cross-Validation (ROC-AUC):')
print('=' * 60)

cv_results = {}
for name, model_template in [
    ('Logistische Regression', LogisticRegression(max_iter=1000, random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))
]:
    scores = cross_val_score(model_template, X_all_scaled, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:30s}: AUC = {scores.mean():.4f} (+/- {scores.std():.4f})')
    print(f'{"":30s}  Folds: {["{:.4f}".format(s) for s in scores]}')

# Boxplot der CV-Ergebnisse
fig, ax = plt.subplots(figsize=(10, 5))
cv_df = pd.DataFrame(cv_results)
cv_df.plot(kind='box', ax=ax, vert=True)
ax.set_title('5-Fold Cross-Validation: ROC-AUC Verteilung')
ax.set_ylabel('ROC-AUC')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Zusammenfassung der Ergebnisse

In [ ]:
print('='*70)
print('              ZUSAMMENFASSUNG DER ERGEBNISSE')
print('='*70)

print('\n--- Hypothesentests ---\n')

print('H1: Je mehr Kontaktversuche (campaign), desto unwahrscheinlicher der Abschluss.')
print('    => STATISTISCH GETESTET: Mann-Whitney-U-Test und Punkt-biseriale Korrelation')
print('    => Negativer Zusammenhang zwischen campaign und Abschluss best\u00e4tigt.')
print('    => Im Modell: campaign_log hat negativen Koeffizienten.\n')

print('H2: Kunden mit Housing UND Loan schlie\u00dfen seltener ab.')
print('    => STATISTISCH GETESTET: Chi\u00b2-Test und z-Test f\u00fcr Proportionen')
print('    => Kunden mit doppelter Kreditbelastung haben niedrigste Abschlussrate.\n')

print('H3: Kunden mit poutcome=success schlie\u00dfen signifikant h\u00e4ufiger ab.')
print('    => STATISTISCH GETESTET: Chi\u00b2-Test und z-Test f\u00fcr Proportionen')
print('    => St\u00e4rkster Effekt im gesamten Datensatz.\n')

print('H4: Management/Rentner schlie\u00dfen h\u00e4ufiger ab als Blue-Collar.')
print('    => STATISTISCH GETESTET: Chi\u00b2-Test und paarweise z-Tests')
print('    => Signifikante Unterschiede zwischen Berufsgruppen best\u00e4tigt.\n')

print('--- Modellperformance ---\n')

best_model_name = max(results.keys(), key=lambda k: results[k]['roc_auc'])
best_result = results[best_model_name]

print(f'Bestes Modell: {best_model_name}')
print(f'  Accuracy:  {best_result["accuracy"]:.4f}')
print(f'  F1-Score:  {best_result["f1_score"]:.4f}')
print(f'  ROC-AUC:   {best_result["roc_auc"]:.4f}')

print('\n--- Hinweise ---\n')
print('- Die Variable "duration" wurde bewusst entfernt (Data Leakage).')
print('- SMOTE wurde verwendet, um die Klassenungleichheit auszugleichen.')
print('- Dies ist ein erster Draft. M\u00f6gliche Verbesserungen:')
print('  * Hyperparameter-Tuning (GridSearchCV / RandomizedSearchCV)')
print('  * Feature Selection (z.B. RFE)')
print('  * Weitere Modelle (XGBoost, LightGBM, SVM)')
print('  * Getrennte Modelle f\u00fcr Erst- und Wiederkontakte')